# 04 aoframe

Question: does AoFrame, the framing of one Homa packet on a byte stream in `homa::stream`, frame,
check and resynchronise as the Homa design note, first version, section 5 states, in its
delimited and its prefixed form.

The side under test is `aosabi-devtool homa-frame`, which frames one packet and pushes a byte stream
through one decoder of the crate per request line. The reference side is
`tests/harness/homa/aoframe.py` with `cobs.py` and `crc32c.py`, written from the design note, the
COBS paper and the CRC catalogue and sharing no code with the crate, and the anchors are the
published vectors of RFC 3720 appendix B.4 and the check value of CRC-32/ISCSI, which
`tests/harness/homa/vectors.py` parsed out of the pinned texts into
`tests/capture/homa_vectors/crc32c.json`.

Eight sections run: the published vectors, COBS runs around the block of 254 bytes, seeded frames
of both forms through both sides, junk ahead of a frame, 100,000 seeded frames with 1 to 5
flipped bits, frames cut and merged, a frame that overflows the buffer, and prefixes above the
buffer. The exit number is that of stage H0 of the plan's section The notebooks: 0 byte
differences, 100 percent of damaged frames dropped, the first good frame after junk accepted, an
overflow dropped up to the next delimiter, and 2 of 2 oversized prefixes refused before a byte is
stored.

Judged stage: H0 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The seed of every drawn frame and every flipped bit.
seed = 20260927

# The frames with flipped bits of the exit number.
max_ops = 100000

# The seeded frames of each form that both sides encode and decode whole.
frames_ops = 2000

# The largest packet of the two delimited links, Serial Line and USB serial,
# and of the prefixed one, a stream Unix Socket.
serial_max_packet = 256
usb_serial_max_packet = 512
stream_max_packet = 16384

# The longest packet drawn for the prefixed form, whose decoder holds
# stream_max_packet bytes. The draw stops at 2,048 bytes so both sides frame
# and read the 2,000 packets within the per call bound, and the section of
# the oversized prefixes checks the length at the buffer's own bound.
prefixed_draw_max = 2048

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the installed targets and the gates of the plan's section The
commands that the notebook rests on: the build of the devtool, the lints of the workspace and the
format of the crates `homa` and `aosabi-devtool`, every cargo call with `--locked`.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import bytediff, paths, run
from harness.homa import aoframe, cobs, crc32c

P = paths(aosabi_root)
OUT = P.out / "04_aoframe"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"
VECTORS = P.tests / "capture" / "homa_vectors" / "crc32c.json"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, and the format of the crate homa under its own edition 2021
# and of the devtool, which rustfmt checks because cargo fmt takes no
# --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-frame", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("vectors        :", P.rel(VECTORS))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "homa-frame", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []
DIFFS = {}


def batch(requests, name):
    """Send `requests` to `homa-frame` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-frame", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    fatal = [a for a in answers if "fatal" in a]
    assert not fatal, "the seam refused requests of %s: %s" % (name, fatal[:3])
    return answers


def encode_both(name, mode, packets):
    """Frame `packets` on both sides and compare the frames byte for byte."""
    answers = batch(
        [{"id": i, "op": "encode", "mode": mode, "hex": p.hex()} for i, p in enumerate(packets)],
        name,
    )
    crate = [a.get("hex", a.get("error")) for a in answers]
    reference = [aoframe.encode(mode, p).hex() for p in packets]
    differing = sum(1 for c, r in zip(crate, reference) if c != r)
    DIFFS[name] = differing
    print("%s: %d frames, %d differ" % (name, len(packets), differing))
    bytediff("\n".join(reference).encode(), "\n".join(crate).encode(), name)
    return crate


def feed_both(name, mode, streams, max_packet):
    """Push each stream through a decoder on both sides and compare every
    frame that ends, its offset and its packet or the name of its drop."""
    answers = batch(
        [{"id": i, "op": "feed", "mode": mode, "hex": s.hex(), "max_packet": max_packet}
         for i, s in enumerate(streams)],
        name,
    )
    crate = []
    for answer in answers:
        crate.append([
            (e["at"], "packet", e["packet"]) if "packet" in e else (e["at"], "error", e["error"])
            for e in answer["events"]
        ])
    reference = [
        [(at, kind, value.hex() if kind == "packet" else value)
         for at, kind, value in aoframe.feed(mode, s, max_packet)]
        for s in streams
    ]
    differing = [i for i, (c, r) in enumerate(zip(crate, reference)) if c != r]
    DIFFS[name] = len(differing)
    print("%s: %d streams, %d whose events differ" % (name, len(streams), len(differing)))
    for i in differing[:3]:
        print("  stream", i, streams[i].hex()[:80])
        print("    crate    ", crate[i][:4])
        print("    reference", reference[i][:4])
    return crate

## The published vectors

`crc32c.json` holds the five examples of RFC 3720 appendix B.4 with the four bytes of each CRC in
the order the RFC prints them, which is the order a delimited frame appends them, and the check
value of CRC-32/ISCSI over the text 123456789. The cell checks the four runs of 32 bytes against
the titles that name them, holds both functions of the reference to the vectors, has the crate
frame each vector and reads the CRC back out of the crate's frame with the reference's COBS, and
compares each frame with the reference's frame byte for byte.

In [ ]:
document = json.loads(VECTORS.read_text(encoding="utf-8"))
for source in document["sources"]:
    print("source:", source["url"], source["sha256"][:16], "read", source["read"])
vectors = [(v["name"], bytes.fromhex(v["data"]), bytes.fromhex(v["crc"])) for v in document["vectors"]]
expected_runs = {
    "32 bytes of zeroes": bytes(32),
    "32 bytes of ones": b"\xff" * 32,
    "32 bytes of incrementing 00..1f": bytes(range(32)),
    "32 bytes of decrementing 1f..00": bytes(range(31, -1, -1)),
}
for name, data, _ in vectors:
    if name in expected_runs:
        assert data == expected_runs[name], "the vector %r does not hold the run its title names" % name
check = document["check"]
check_text = check["check_text"].encode()
check_value = int(check["check"], 16)
cases = vectors + [("check value of CRC-32/ISCSI", check_text, check_value.to_bytes(4, "little"))]
frames = encode_both("vectors", "delimited", [data for _, data, _ in cases])
print()
print("%-40s %-10s %-10s %-10s %-10s" % ("vector", "published", "reference", "bitwise", "crate"))
wrong = 0
for (name, data, published), frame in zip(cases, frames):
    body = bytes.fromhex(frame)
    assert body[0] == 0 and body[-1] == 0, "a delimited frame opens and ends with a zero byte"
    crate_crc = cobs.decode(body[1:-1])[-4:]
    reference_crc = crc32c.appended(data)
    bitwise_crc = crc32c.crc32c_bitwise(data).to_bytes(4, "little")
    same = published == reference_crc == bitwise_crc == crate_crc
    wrong += not same
    print("%-40s %-10s %-10s %-10s %-10s%s" % (
        name[:40], published.hex(), reference_crc.hex(), bitwise_crc.hex(), crate_crc.hex(),
        "" if same else "  DIFFERS"))
DIFFS["vector crcs"] = wrong
print()
print("check value over %r: 0x%08x" % (check["check_text"], int.from_bytes(crate_crc, "little")))
assert int.from_bytes(crate_crc, "little") == 0xE3069283 == check_value

## COBS around the block of 254 bytes

A block of COBS holds at most 254 bytes that are not zero, so the runs of 253, 254 and 255 such
bytes, each with and without a zero byte behind it, are the boundaries of the encoding. A block of
254 bytes that ends the input is written without the code 1 of an empty block after it. The crate
and the reference frame each run, and each side decodes the other's frame.

In [ ]:
runs = []
for length in (253, 254, 255, 508, 509):
    for tail in (b"", b"\x00", b"\x00\x07"):
        runs.append(bytes((i % 255) + 1 for i in range(length)) + tail)
runs.append(b"")
runs.append(b"\x00")
runs.append(b"\x00" * 300)
cobs_frames = encode_both("cobs runs", "delimited", runs)
cobs_decoded = feed_both("cobs runs back", "delimited", [bytes.fromhex(f) for f in cobs_frames], 1024)
back = sum(1 for events, packet in zip(cobs_decoded, runs) if events == [(len(aoframe.encode("delimited", packet)) - 1, "packet", packet.hex())])
print("runs the crate reads back whole from its own frames: %d of %d" % (back, len(runs)))
assert back == len(runs)

## Seeded frames of both forms

The seeded packets run from 1 byte to the largest packet of each delimited link, 256 bytes on the
Serial Line and 512 on USB serial, and to `prefixed_draw_max`, 2,048 bytes, on the prefixed
stream, whose decoder holds a packet of 16,384 bytes. A quarter of them are Japanese text in UTF-8
and the rest random bytes with the zero byte as common as any other. Each form
frames them on both sides, and each side's frames, one after the other on one stream, go through
a decoder on both sides, whose events must be equal.

In [ ]:
JAPANESE = "日本語のテキストと、漢字、ひらがな、カタカナ。".encode()
g = random.Random(seed)


def seeded_packet(max_packet):
    length = g.randint(1, max_packet)
    if g.random() < 0.25:
        return (JAPANESE * (length // len(JAPANESE) + 1))[:length]
    return bytes(g.getrandbits(8) for _ in range(length))


for mode, max_packet, drawn in (("delimited", serial_max_packet, serial_max_packet),
                                ("delimited", usb_serial_max_packet, usb_serial_max_packet),
                                ("prefixed", stream_max_packet, prefixed_draw_max)):
    packets = [seeded_packet(drawn) for _ in range(frames_ops)]
    label = "%s %d up to %d" % (mode, max_packet, drawn)
    crate_frames = encode_both(label + " encode", mode, packets)
    chunks = [b"".join(bytes.fromhex(f) for f in crate_frames[i:i + 100]) for i in range(0, len(crate_frames), 100)]
    events = feed_both(label + " stream", mode, chunks, max_packet)
    accepted = [bytes.fromhex(value) for stream in events for _, kind, value in stream if kind == "packet"]
    print("  packets back from the crate's stream: %d of %d, equal: %s" % (len(accepted), len(packets), accepted == packets))
    assert accepted == packets

## Junk ahead of a frame

A modem or a boot loader writes text on a serial line before the link carries frames. The junk ends
at the zero byte that opens the first frame, so the decoder drops it as one damaged frame and
accepts the frame after it. The streams hold AT commands, a boot banner and random bytes ahead of
one good frame.

In [ ]:
good = JAPANESE[:40]
junk_heads = [
    b"AT\r\n",
    b"AT+CGMI\r\nAT+CGMM\r\nATI\r\n",
    b"ESP-ROM:esp32c3-api1-20210207\r\nBuild:Feb  7 2021\r\n",
    b"+++",
    bytes(g.getrandbits(8) | 1 for _ in range(200)),
]
streams = [head + aoframe.encode("delimited", good) for head in junk_heads]
junk_events = feed_both("junk", "delimited", streams, serial_max_packet)
first_good = 0
for head, events in zip(junk_heads, junk_events):
    packets = [value for _, kind, value in events if kind == "packet"]
    dropped = [value for _, kind, value in events if kind == "error"]
    first_good += packets[:1] == [good.hex()]
    print("junk of %3d bytes: dropped %s, then %d packet, the good frame: %s" % (
        len(head), dropped, len(packets), packets[:1] == [good.hex()]))
print("the first good frame after junk accepted: %d of %d" % (first_good, len(junk_heads)))

## 100,000 frames with flipped bits

Each seeded frame carries a packet of 20 to 320 bytes, the range of the Serial Line and USB serial
links, and has 1 to 5 of its bits flipped at seeded places, the delimiters included, and each
damaged frame is its own stream. A damaged frame is dropped when its stream yields no packet: the
decoder refuses it as `Cobs` or `Crc`, or a flipped delimiter leaves the frame without an end.

In [ ]:
flip = random.Random(seed ^ 0x04)
damaged = []
flips = []
for _ in range(max_ops):
    length = flip.randint(20, 320)
    packet = bytes(flip.getrandbits(8) for _ in range(length))
    frame = bytearray(aoframe.encode("delimited", packet))
    count = flip.randint(1, 5)
    places = flip.sample(range(len(frame) * 8), count)
    for place in places:
        frame[place // 8] ^= 1 << (place % 8)
    damaged.append(bytes(frame))
    flips.append(count)
flip_events = feed_both("flipped", "delimited", damaged, usb_serial_max_packet)
accepted_damaged = sum(1 for events in flip_events if any(kind == "packet" for _, kind, _ in events))
names = {}
for events in flip_events:
    for _, kind, value in events:
        if kind == "error":
            names[value] = names.get(value, 0) + 1
unended = sum(1 for events in flip_events if not events)
print("damaged frames:", len(damaged), " by flipped bits:", {n: flips.count(n) for n in range(1, 6)})
print("damaged frames that yielded a packet:", accepted_damaged)
print("drops by name:", names, " streams with no end of frame:", unended)
dropped_share = 100.0 * (len(damaged) - accepted_damaged) / len(damaged)
print("damaged frames dropped: %.3f percent" % dropped_share)

## Frames cut and merged

A lost byte, a merged frame, whose delimiter between two frames was lost, and a split frame, which a
stray zero byte cuts in two, fit the buffer and fail the COBS decode or the CRC, whichever check
meets the damage first. Each stream carries the damaged frame between two good ones, so it also
shows the decoder in step again after the damage.

In [ ]:
cut = random.Random(seed ^ 0x05)
cut_streams = []
kinds = []
for n in range(300):
    first = aoframe.encode("delimited", bytes(cut.getrandbits(8) for _ in range(cut.randint(20, 120))))
    second = aoframe.encode("delimited", bytes(cut.getrandbits(8) for _ in range(cut.randint(20, 120))))
    before = aoframe.encode("delimited", b"before")
    after = aoframe.encode("delimited", b"after")
    kind = n % 3
    if kind == 0:
        at = cut.randrange(2, len(first) - 2)
        middle = first[:at] + first[at + 1:]
    elif kind == 1:
        middle = first[:-1] + second[1:]
    else:
        at = cut.randrange(2, len(first) - 2)
        middle = first[:at] + b"\x00" + first[at:]
    cut_streams.append(before + middle + after)
    kinds.append(("lost byte", "merged", "split")[kind])
cut_events = feed_both("cut and merged", "delimited", cut_streams, usb_serial_max_packet)
wrong_packets = 0
in_step = 0
for events in cut_events:
    packets = [value for _, kind, value in events if kind == "packet"]
    wrong_packets += sum(1 for p in packets if p not in (b"before".hex(), b"after".hex()))
    in_step += packets[:1] == [b"before".hex()] and packets[-1:] == [b"after".hex()]
print("streams:", len(cut_streams), {k: kinds.count(k) for k in set(kinds)})
print("packets out of a damaged frame:", wrong_packets)
print("streams whose good frames on both sides arrived: %d of %d" % (in_step, len(cut_streams)))

## A frame that overflows the buffer

A delimited frame longer than the buffer of the link is dropped as `TooLong` at the delimiter that
ends it, and the frame after it is read as usual. The stream carries a frame of the largest
packet plus 50 bytes and then a good frame, on a decoder sized for the Serial Line.

In [ ]:
long_frame = aoframe.encode("delimited", bytes(range(1, 256)) + bytes(range(1, 52)))
overflow_stream = long_frame + aoframe.encode("delimited", good)
overflow_events = feed_both("overflow", "delimited", [overflow_stream], serial_max_packet)[0]
print("buffer of the Serial Line decoder:", aoframe.buffer_len("delimited", serial_max_packet))
print("frame of %d bytes, events:" % len(long_frame), overflow_events)
overflow_ok = (
    overflow_events[:1] == [(len(long_frame) - 1, "error", "TooLong")]
    and overflow_events[1:] == [(len(overflow_stream) - 1, "packet", good.hex())]
)
print("the overflow dropped up to the next delimiter and the next frame read:", overflow_ok)

## Prefixes above the buffer

A prefixed decoder refuses a length above its buffer once the fourth byte of the prefix arrives,
before a byte of the packet is stored, because a length from a peer is an allocation request, and it
skips the bytes of the refused packet so the stream stays in step. The two prefixes are `max_packet`
plus 1 and 2 to the 32nd minus 1 on a decoder of the stream Unix Socket's 16,384 bytes. The first
stream carries the refused packet's bytes and a good frame behind it, and the second carries the
prefix alone, since 4 GiB of bytes are no part of a notebook. The reference's decoder reports the
bytes it stored at the refusal.

In [ ]:
over = stream_max_packet + 1
prefixed_streams = [
    over.to_bytes(4, "big") + bytes(over) + aoframe.encode("prefixed", good),
    (2 ** 32 - 1).to_bytes(4, "big"),
]
prefix_events = feed_both("prefixes", "prefixed", prefixed_streams, stream_max_packet)
refused_before = 0
for stream, events in zip(prefixed_streams, prefix_events):
    decoder = aoframe.Decoder("prefixed", stream_max_packet)
    stored = None
    for byte in stream[:4]:
        if decoder.push(byte) is not None:
            stored = decoder.stored
    first = events[:1] == [(3, "error", "TooLong")]
    refused_before += first and stored == 0
    print("prefix %d: first event %s, bytes stored at the refusal %s, events %s" % (
        int.from_bytes(stream[:4], "big"), events[:1], stored, events[1:]))
skip_ok = prefix_events[0][1:] == [(len(prefixed_streams[0]) - 1, "packet", good.hex())]
print("oversized prefixes refused before a byte is stored: %d of 2" % refused_before)
print("the good frame behind the skipped packet read:", skip_ok)

## Verdict

In [ ]:
print("differences by section:", DIFFS)
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
assert not PANICS, PANICS
assert sum(DIFFS.values()) == 0, DIFFS
assert accepted_damaged == 0 and dropped_share == 100.0
assert first_good == len(junk_heads)
assert wrong_packets == 0 and in_step == len(cut_streams)
assert overflow_ok and skip_ok
assert refused_before == 2
print()
print("VERDICT: 0 byte differences on the vectors of RFC 3720 appendix B.4 and the check value"
      " 0x%08x, %.0f percent of %d damaged frames dropped, the first good frame after junk"
      " accepted %d of %d times, an overflow dropped up to the next delimiter, and %d of 2"
      " oversized prefixes refused before a byte is stored"
      % (check_value, dropped_share, len(damaged), first_good, len(junk_heads), refused_before))